# EXP-012 — Đổi LLM teacher BLIP-2/FlanT5 sang Qwen2.5-VL-7B
Giữ nguyên CLIP L/14 và head tuyến tính; chỉ đổi **vector target**: hidden state của Qwen2.5-VL-7B-Instruct (3584-d) khi đọc ảnh mặt 224×224 (64 token ảnh) + instruction gốc. Chỉ forward 1 lần, không generate.

1. Tính target Qwen cho CAER-S (A100/L4, chạy tiếp được nếu bị ngắt). 4 cách lấy vector: `all` (mặc định, mean-pool lượt user ở lớp cuối), `img`, `last`, `mid_all` (lớp 2/3 độ sâu).
2. Train từ cache (như EXP-003): BLIP-2 (target cũ `exp001/full`) và 4 biến thể Qwen × 3 seed, lưu e5/e10.
3. Test trong notebook (P0, P0+flip) + 1 lần `test.py` để đối chiếu.

**Tiêu chí (chọn trước):** Qwen `all` e5 − BLIP-2 e5 (cùng seed, cùng pipeline), Ensemble Five P0: Δ Mean4 UAR ≥ +1.0 và không dataset nào < −1.0. `img`/`last`/`mid_all` chỉ là ablation.

# Cấu hình

In [ ]:
QWEN_MODEL = "Qwen/Qwen2.5-VL-7B-Instruct"
SEEDS = [1, 2, 3]
EPOCHS = 10
SAVE_EPOCHS = [5, 10]
LR = 1e-3
POOLS = ["all", "img", "last", "mid_all"]
VARIANTS = {"blip2": ("BLIP2", "all")} | {f"qwen_{p}": ("QWEN", p) for p in POOLS}   # tên → (nguồn target, pool)
job_of = lambda v, seed: f"exp012_{v}_l14_seed{seed}"
BRANCH = "exp/012-qwen-teacher"

INSTRUCTION = "Please play the role of a facial action describer. Objectively describe the detailed facial actions of the person in the image."
TRAIN_DATA = "/content/CAER-S_Face/train"
FACE_ZIP = "/content/drive/MyDrive/Datasets/CAERS_Face_train.zip"
TEST_ZIP = "/content/drive/MyDrive/Datasets/static_test_sets.zip"
OUT_DIR = "/content/drive/MyDrive/ExpCLIP_output/exp012"
BLIP2_DIR = "/content/drive/MyDrive/ExpCLIP_output/exp001/full"   # target BLIP-2 + feature CLIP CAER-S (EXP-001/003)
CLIP_CACHE = f"{BLIP2_DIR}/clip_feats_l14.npz"
QWEN_DIR = f"{OUT_DIR}/pre_qwen7b"                                   # targets/chunk_*.npz
QWEN_DONE = f"{QWEN_DIR}/targets_done.txt"
TARGET_DIR = {"BLIP2": f"{BLIP2_DIR}/targets", "QWEN": f"{QWEN_DIR}/targets"}
EXP000_ROOT = "/content/drive/MyDrive/ExpCLIP_output"

import os, subprocess
def run(cmd):
    """Chạy lệnh shell, in output trực tiếp; lỗi thì dừng cell (khác với `!`)."""
    # bash + pipefail: lệnh có `| tee` vẫn báo lỗi nếu python lỗi
    p = subprocess.Popen("set -o pipefail; " + cmd, shell=True, executable="/bin/bash", text=True,
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         env={**os.environ, "PYTHONUNBUFFERED": "1"})  # in log ngay, không đợi hết
    for line in p.stdout:
        print(line, end="")
    if p.wait() != 0:
        raise RuntimeError(f"Lệnh lỗi (exit {p.returncode}) — xem output ở trên")

# Mount Drive & GPU
Chưa tính xong target Qwen thì cần GPU ≥ 24GB (A100 hoặc L4; Qwen2.5-VL-7B bf16 ≈ 17GB). Xong rồi thì GPU nào cũng được.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
os.makedirs(QWEN_DIR, exist_ok=True)
!nvidia-smi --query-gpu=name,memory.total --format=csv
NEED_QWEN = not os.path.exists(QWEN_DONE)
gpu_mem = int(subprocess.check_output("nvidia-smi --query-gpu=memory.total --format=csv,noheader,nounits", shell=True).split()[0])
if NEED_QWEN:
    assert gpu_mem >= 22000, f"GPU chỉ có {gpu_mem} MiB — cần A100/L4 để chạy Qwen2.5-VL-7B (Runtime > Change runtime type)"
assert os.path.exists(CLIP_CACHE) and os.path.isdir(TARGET_DIR["BLIP2"]), "thiếu cache CAER-S của EXP-001/003"
print("Cần Qwen:", NEED_QWEN)

# Clone repo & giải nén data

In [ ]:
if not os.path.isdir("/content/Exp-CLIP"):
    !git clone -b $BRANCH https://github.com/huongng389/Exp-CLIP.git /content/Exp-CLIP
%cd /content/Exp-CLIP
assert os.getcwd() == "/content/Exp-CLIP", "clone lỗi — nhánh đã push chưa?"
!git checkout $BRANCH && git pull
!git log --oneline -1   # ghi lại commit vào research_log
!mkdir -p /content/CAER-S_Face /content/static_test_sets
!unzip -q -n "$FACE_ZIP" -d /content/CAER-S_Face
!unzip -q -n "$TEST_ZIP" -d /content/static_test_sets
!ls /content/CAER-S_Face/train

# 1) Target Qwen2.5-VL-7B (môi trường Colab mặc định)
Ghi từng chunk vào Drive → bị ngắt thì chạy lại cell này, nó tiếp từ chỗ dừng. Dòng đầu log in số token ảnh (phải là 64) và số chiều (3584).

In [ ]:
if NEED_QWEN:
    !pip install -q -U "transformers>=4.51" accelerate
    run(f'python tools/precompute_qwen_targets.py --train-data {TRAIN_DATA} --out-dir {QWEN_DIR} '
        f'--model {QWEN_MODEL} --batch-size 32 --workers 8 --instruction "{INSTRUCTION}"')
    open(QWEN_DONE, "w").write("ok")
!ls "$QWEN_DIR/targets" | wc -l

# Chẩn đoán target (trước khi train)
- **cos 2 ảnh**: cos trung bình giữa target của 2 ảnh ngẫu nhiên khác nhau. Thấp hơn → target đa dạng hơn.
- **Map tuyến tính từ CLIP**: hồi quy ridge feature CLIP → target trên 90% ảnh, đo cos trên 10% còn lại. Head chỉ là 1 lớp tuyến tính, nên target khó map tuyến tính thì khó học.

In [ ]:
import glob, numpy as np
def load_targets(d, pool):
    P, X = [], []
    for c in sorted(glob.glob(f"{d}/chunk_*.npz")):
        z = np.load(c); P += list(z["paths"]); X.append(z[pool].astype(np.float32))
    return dict(zip(P, np.concatenate(X)))
def pair_cos(x, n=5000, seed=0):
    rng = np.random.default_rng(seed); a, b = rng.integers(0, len(x), (2, n))
    return float(((x[a] * x[b]).sum(1))[a != b].mean())

clip_npz = np.load(CLIP_CACHE)
clip_feat = dict(zip(clip_npz["paths"], clip_npz["feats"].astype(np.float32)))
diag = {}
for v, (src, pool) in VARIANTS.items():
    t = load_targets(TARGET_DIR[src], pool)
    keys = sorted(set(t) & set(clip_feat))
    T = np.stack([t[k] for k in keys]); X = np.stack([clip_feat[k] for k in keys])
    T = T / np.linalg.norm(T, axis=1, keepdims=True)   # lưu fp16 → chuẩn hoá lại
    X = X / np.linalg.norm(X, axis=1, keepdims=True)
    idx = np.random.default_rng(0).permutation(len(keys)); cut = int(0.9 * len(keys))
    tr, te = idx[:cut], idx[cut:]
    W = np.linalg.solve(X[tr].T @ X[tr] + 1e-2 * np.eye(X.shape[1]), X[tr].T @ T[tr])
    pred = X[te] @ W; pred /= np.linalg.norm(pred, axis=1, keepdims=True)
    Tc = T - T[tr].mean(0)   # bỏ phần chung mọi ảnh (cùng instruction) → đo phần riêng từng ảnh
    pc = X[te] @ np.linalg.solve(X[tr].T @ X[tr] + 1e-2 * np.eye(X.shape[1]), X[tr].T @ Tc[tr])
    r2 = 1 - ((Tc[te] - pc) ** 2).sum() / (Tc[te] ** 2).sum()
    diag[v] = {"n": len(keys), "dim": T.shape[1], "cos 2 ảnh": pair_cos(T), "cos dự đoán (test 10%)": float((pred * T[te]).sum(1).mean()),
               "R² phần riêng": float(r2)}
import pandas as pd
print(pd.DataFrame(diag).T.round(4).to_string())
del clip_feat

# Môi trường Python 3.10 (lavis)
`train.py`/`test.py` import `models/Exp_CLIP.py` → cần lavis (dù train từ cache không chạy BLIP-2).

In [ ]:
!sudo apt-get -qq update > /dev/null
!sudo apt-get -qq install -y python3.10 python3.10-venv python3.10-distutils > /dev/null
!python3.10 -m venv /content/venv310
!/content/venv310/bin/pip install -q --upgrade pip
!/content/venv310/bin/pip install -q --no-cache-dir torch==2.0.1 torchvision==0.15.2 torchaudio==2.0.2 --index-url https://download.pytorch.org/whl/cu118
!/content/venv310/bin/pip install -q --no-cache-dir transformers tqdm regex pyyaml pandas pydantic click timm einops
!/content/venv310/bin/pip install -q --no-cache-dir salesforce-lavis
# phải cài SAU lavis: lavis kéo numpy 2.x / opencv mới
!/content/venv310/bin/pip install -q --no-cache-dir "numpy<2" "opencv-python==4.10.0.84" "opencv-python-headless==4.10.0.84"
!/content/venv310/bin/pip install -q --no-cache-dir matplotlib scikit-learn
!/content/venv310/bin/python -c "import numpy, torch, lavis, sklearn; print('numpy', numpy.__version__, '| torch', torch.__version__, '| cuda', torch.cuda.is_available())"
# môi trường Colab mặc định (dùng cho phần test trong notebook)
!pip install -q ftfy "setuptools<81"

# 2) Train từ cache: 5 biến thể × 3 seed

In [ ]:
for seed in SEEDS:
  for v, (src, pool) in VARIANTS.items():
    job = job_of(v, seed)
    if os.path.exists(f"{OUT_DIR}/{job}-e{SAVE_EPOCHS[-1]}-model.pth"):
        print(f"{v} seed {seed}: đã train xong, bỏ qua"); continue
    resume = f"{OUT_DIR}/{job}-resume.pth"
    print(f"=============== {v}, seed {seed}")
    run(f'/content/venv310/bin/python train.py '
        f'--epochs {EPOCHS} --batch-size 512 --batch-size-test-image 512 --batch-size-test-video 64 '
        f'--lr {LR} --weight-decay 1e-4 --momentum 0.9 --print-freq 1000 --milestones 1000 --seed {seed} '
        f'--job-id {job} --instruction "{INSTRUCTION}" --load-model CLIP_L14 --train-data {TRAIN_DATA} '
        f'--cache-clip {CLIP_CACHE} --cache-targets {TARGET_DIR[src]} --cache-pool {pool} '
        f'--save-epochs {" ".join(map(str, SAVE_EPOCHS))} --extra-checkpoint-dir {OUT_DIR}'
        + (f' --resume {resume}' if os.path.exists(resume) else ''))

In [ ]:
for seed in SEEDS:
  for v in VARIANTS:
    lines = open(f"{OUT_DIR}/{job_of(v, seed)}-log.txt").read().splitlines()
    acc = [float(l.split(":")[1]) for l in lines if l.startswith("The best accuracy")]
    print(f"{v} seed {seed}:", [l for l in lines if l.startswith("Cached training")][:1],
          "| train acc", {e: round(acc[e - 1], 2) for e in SAVE_EPOCHS if e <= len(acc)})

# 3) Đối chiếu bằng `test.py` (Qwen `all` seed 1 e5)

In [ ]:
CHECK_JOB = f"{job_of('qwen_all', 1)}-e5"
!mkdir -p checkpoint log
if not os.path.exists(f"{OUT_DIR}/{CHECK_JOB}-test.txt"):
    !cp "$OUT_DIR/$CHECK_JOB-model.pth" checkpoint/
    run(f'/content/venv310/bin/python test.py --load-model CLIP_L14 --job-id {CHECK_JOB} '
        f'--data-root /content/static_test_sets 2>&1 | tee log/{CHECK_JOB}-test.txt')
    !cp log/$CHECK_JOB-test.txt "$OUT_DIR/"
txt_log = open(f"{OUT_DIR}/{CHECK_JOB}-test.txt").read()
print(txt_log[-1500:])

# 4) Test trong notebook
Feature ảnh test P0 × (gốc, lật) tính 1 lần; mỗi head chỉ là phép nhân ma trận (như EXP-008/010).

In [ ]:
import sys, re, json, torch, torch.nn.functional as F
import torchvision.datasets as datasets
sys.path.insert(0, "/content/Exp-CLIP")
from models.clip import clip
from models.Text import FER_prompt_list, FER_prompt_type_list
from data_loader.image_transform import image_transform

DATASETS = ["RAFDB", "AffectNet7", "AffectNet8", "FERPlus"]
N_CLS = {"RAFDB": 7, "AffectNet7": 7, "AffectNet8": 8, "FERPlus": 8}
PROMPT_NAMES = ["Expression Ensemble Five", "Expression Ensemble Ten"]
BASE_PROMPT = "Expression Ensemble Five"
CFG_NAMES = ["P0", "P0+flip"]

def find_ckpt(root, name):
    hits = glob.glob(f"{root}/**/{name}", recursive=True)
    return hits[0] if hits else None

HEADS = {
    "CLIP gốc": {0: None},
    "Paper": {0: "/content/Exp-CLIP/checkpoint/ExpCLIP_L14_model.pth"},
    "EXP-000": {s: find_ckpt(EXP000_ROOT, f"expclip_l14_face_seed{s}-model.pth") for s in SEEDS},
} | {f"{v} e{e}": {s: f"{OUT_DIR}/{job_of(v, s)}-e{e}-model.pth" for s in SEEDS} for v in VARIANTS for e in SAVE_EPOCHS}
for m, d in HEADS.items():
    for s, p in list(d.items()):
        if p is not None and not os.path.exists(p):
            print(f"CẢNH BÁO: {m} seed {s}: không thấy checkpoint ({p}) → bỏ qua")
            del d[s]

PROMPTS = {pn: pl for pn, pl in zip(FER_prompt_type_list, FER_prompt_list) if pn in PROMPT_NAMES}
clip_model, _ = clip.load("ViT-L/14", "cuda")
clip_model.eval()
img, y, classes, txt = {}, {}, {}, {}   # img[ds] = (feat gốc, feat lật)
with torch.no_grad():
    for ds_name in DATASETS:
        ds = datasets.ImageFolder(f"/content/static_test_sets/{ds_name}/test", image_transform("raw"))
        loader = torch.utils.data.DataLoader(ds, batch_size=256, shuffle=False, num_workers=4)
        f0, f1 = [], []
        for x, _ in loader:
            x = x.cuda()
            f0.append(clip_model.encode_image(x).float().cpu())
            f1.append(clip_model.encode_image(x.flip(-1)).float().cpu())   # lật ngang
        img[ds_name] = (torch.cat(f0), torch.cat(f1))
        y[ds_name] = np.array(ds.targets)
        classes[ds_name] = [c.split("_", 1)[1] for c in ds.classes]   # 0_Happiness → Happiness
        for pn, pl in PROMPTS.items():
            txt[(ds_name, pn)] = clip_model.encode_text(
                clip.tokenize(pl[ds_name], context_length=77, truncate=True).cuda()).float().cpu()
        print(ds_name, tuple(img[ds_name][0].shape), classes[ds_name])

In [ ]:
def load_W(path):
    if path is None:
        return None
    sd = torch.load(path, map_location="cpu")
    return next(v for k, v in sd.items() if "mlp.weight" in k).float()   # (4096 hoặc 3584, 768)

def predict(W, ds_name, cfg, pn):
    f0, f1 = img[ds_name]
    i, t = ((f0 + f1) / 2 if cfg.endswith("+flip") else f0), txt[(ds_name, pn)]   # TTA: trung bình trước head
    if W is not None:
        i, t = i @ W.T, t @ W.T
    i, t = F.normalize(i, dim=-1), F.normalize(t, dim=-1)
    return (i @ t.T).view(len(i), N_CLS[ds_name], -1).mean(-1).argmax(1).numpy()   # prompt xếp theo lớp, như test.py

def metrics(y_, p_, n_cls):
    acc = np.array([(p_[y_ == c] == c).mean() * 100 for c in range(n_cls)])
    return {"UAR": acc.mean(), "WAR": (p_ == y_).mean() * 100, "acc": acc}

rows = []
for m, d in HEADS.items():
    for s, path in d.items():
        W = load_W(path)
        for ds_name in DATASETS:
            for cfg in CFG_NAMES:
                for pn in PROMPT_NAMES:
                    r = metrics(y[ds_name], predict(W, ds_name, cfg, pn), N_CLS[ds_name])
                    rows.append({"model": m, "seed": s, "dataset": ds_name, "config": cfg, "prompt": pn,
                                 "UAR": r["UAR"], "WAR": r["WAR"],
                                 **{f"acc_{c}": a for c, a in zip(classes[ds_name], r["acc"])}})
res = pd.DataFrame(rows)
res.to_csv(f"{OUT_DIR}/exp012_qwen_teacher-results.csv", index=False)
print(len(res), "dòng →", OUT_DIR)

# Kiểm tra pipeline
(a) EXP-000 phải khớp số đã ghi (`test.py`). (b) Qwen `all` seed 1 e5 phải khớp `test.py` ở bước 3. Lệch ≤ ~0.4 là do fp16.

In [ ]:
REF = {"EXP-000|1|RAFDB": [58.88, 66.88], "EXP-000|1|AffectNet7": [44.09, 44.09], "EXP-000|1|AffectNet8": [39.23, 39.23], "EXP-000|1|FERPlus": [40.65, 51.47], "EXP-000|2|RAFDB": [57.82, 65.78], "EXP-000|2|AffectNet7": [43.69, 43.69], "EXP-000|2|AffectNet8": [38.17, 38.17], "EXP-000|2|FERPlus": [41.08, 53.6], "EXP-000|3|RAFDB": [57.5, 63.3], "EXP-000|3|AffectNet7": [44.74, 44.74], "EXP-000|3|AffectNet8": [39.12, 39.12], "EXP-000|3|FERPlus": [42.93, 55.16]}

blk = re.split(r"Prompt Type:\s+", txt_log)
blk = next(b for b in blk if b.startswith(BASE_PROMPT + "\n"))
for ds_name in DATASETS:
    mm = re.search(rf"\* {ds_name}\nUAR/WAR: ([\d.]+)/([\d.]+)", blk)
    if mm:
        REF[f"qwen_all e5|1|{ds_name}"] = [float(mm.group(1)), float(mm.group(2))]

chk = []
for k, (u_, w_) in REF.items():
    m, s, ds_name = k.split("|")
    r = res[(res.model == m) & (res.seed == int(s)) & (res.dataset == ds_name) & (res.config == "P0") & (res.prompt == BASE_PROMPT)]
    if len(r):
        chk.append({"key": k, "dUAR": r.UAR.item() - u_, "dWAR": r.WAR.item() - w_})
chk = pd.DataFrame(chk)
print(f"{len(chk)}/{len(REF)} cấu hình so được | max |ΔUAR| = {chk.dUAR.abs().max():.2f}, max |ΔWAR| = {chk.dWAR.abs().max():.2f}")
print(chk.round(2).sort_values("dUAR", key=abs, ascending=False).head(6).to_string(index=False))

# Bảng 1 — UAR (mean ± std qua seed)

In [ ]:
pd.set_option("display.width", 250)
def fmt(x):
    return f"{x.mean():.2f}" if len(x) == 1 else f"{x.mean():.2f} ± {x.std(ddof=1):.2f}"

u = res.pivot_table(index=["model", "seed", "prompt", "config"], columns="dataset", values="UAR")[DATASETS]
u["Mean4"] = u[DATASETS].mean(1)

def get(m, cfg, pn=BASE_PROMPT):
    return u.loc[m].xs(pn, level="prompt").xs(cfg, level="config")   # index: seed

for pn in PROMPT_NAMES:
    t = pd.DataFrame({f"{m} @{c}": get(m, c, pn).agg(fmt) for m in HEADS if HEADS[m] for c in CFG_NAMES}).T
    print(f"--- {pn}\n{t.to_string()}\n")

# Bảng 2 — Tiêu chí (chênh lệch theo cặp seed, Ensemble Five, so với BLIP-2 cùng epoch)

In [ ]:
def compare(a, b, pn=BASE_PROMPT):
    d = get(*a, pn) - get(*b, pn)
    return d.agg(fmt), d.mean()

checks = {"CHÍNH: qwen_all e5 − blip2 e5": (("qwen_all e5", "P0"), ("blip2 e5", "P0"))}
checks |= {f"{v} e{e} − blip2 e{e}": ((f"{v} e{e}", "P0"), (f"blip2 e{e}", "P0"))
           for v in VARIANTS if v != "blip2" for e in SAVE_EPOCHS if (v, e) != ("qwen_all", 5)}
checks |= {"blip2 e5 − EXP-000 (kiểm cache)": (("blip2 e5", "P0"), ("EXP-000", "P0"))}
t, verdict = {}, {}
for name, (a, b) in checks.items():
    t[name], mean = compare(a, b)
    if name.startswith("CHÍNH"):
        ok = mean["Mean4"] >= 1.0 and (mean[DATASETS] >= -1.0).all()
        verdict[name] = (f"Δ Mean4 {mean['Mean4']:+.2f}, min Δ dataset {mean[DATASETS].min():+.2f} → {'ĐẠT' if ok else 'KHÔNG đạt'}")
print(pd.DataFrame(t).T.to_string(), "\n")
for k, v in verdict.items():
    print(f"{k}: {v}")

# Bảng 3 — Accuracy từng lớp (Ensemble Five, P0, mean seed)

In [ ]:
for ds_name in DATASETS:
    cols = [f"acc_{c}" for c in classes[ds_name]]
    t = {}
    for m in ["EXP-000", "blip2 e5"] + [f"qwen_{p} e5" for p in POOLS]:
        r = res[(res.model == m) & (res.dataset == ds_name) & (res.prompt == BASE_PROMPT) & (res.config == "P0")]
        t[m] = r[cols].mean().values
    t = pd.DataFrame(t, index=classes[ds_name]).T
    print(f"--- {ds_name}\n{t.round(1).to_string()}\n")

# Lưu JSON (cùng định dạng `research_log/results/`)

In [ ]:
out = {"source": "notebooks/EXP-012_qwen_teacher.ipynb", "diagnostics": diag,
       "rows": [{"model": r.model, "seed": int(r.seed), "config": r.config, "prompt": r.prompt, "dataset": r.dataset,
                 "UAR": round(r.UAR, 2), "WAR": round(r.WAR, 2)} for r in res.itertuples()]}
with open(f"{OUT_DIR}/EXP-012-qwen-teacher.json", "w") as f:
    json.dump(out, f, indent=1, ensure_ascii=False)
print(f"{len(out['rows'])} dòng → {OUT_DIR}/EXP-012-qwen-teacher.json")